In [ ]:
#!/usr/bin/env python3
"""
v5 G0 — VCC leg, HEADLINE analysis. Engineer, 2026-07-19, per Brian's ruling.

Supersedes the global/specific decomposition on VCC. Brian's noise model,
corr(g^A,g^B) = |gamma|^2 / (|gamma|^2 + N/P), fit against the diagnostics in
v5g0_04_vcc_diagnostics_v1.py, puts VCC's shared program at gamma ~ 0 — 488x
weaker than the weakest single knockdown measured, 14,322x weaker than the
strongest (TADA1). With gamma ~ 0, s_p = d_p - g ~ d_p: the decomposition is
degenerate here, so corr(d_p^A, d_p^B) IS the specific reproducibility. No
subtraction, no stratification-then-evaluation-on-the-same-cells circularity
(the bug flagged 2026-07-18).

NEW HEADLINE QUESTION (Brian, 2026-07-19):
  "What fraction of VCC's 150 perturbations carry a reproducible signal at the
  depth the benchmark provides?"

METHOD
  For each of VCC's 150 perturbations, use its OWN full achievable half-depth
  (not a fixed grid value — "the depth the benchmark provides" is whatever
  cells that perturbation actually has, matching real competition conditions
  where cells-per-perturbation is highly non-uniform, CV=0.71). Compute
  corr(d_p^A, d_p^B) directly against the control's matched-depth profile.
  Also report the fixed-n=600 matched-44 panel as a secondary (comparable
  across perturbations at identical depth, at the cost of dropping to 44).

  Three-way split discipline (Brian, 2026-07-19): halves 0/1 (A/B) are used for
  measurement ONLY. Half 2 (panel C) is reserved for effect-size ranking and is
  NEVER used to select perturbations that are then evaluated on A/B.

INPUT
  results/v5 G0 - VCC/v5g0_vcc_accum_v1.npz
OUTPUT
  results/v5 G0 - VCC/v5g0_vcc_direct_specific_v1.csv   (per-perturbation, per-seed)
  results/v5 G0 - VCC/v5g0_vcc_direct_specific_summary_v1.json
"""
import json
import os

import numpy as np
import pandas as pd

# Two supported layouts:
#   sandbox (default) - reads/writes straight to the repo's results/ folder.
#   Kaggle - set V5G0_VCC_ACCUM / V5G0_OUT, or attach a Dataset containing
#     v5g0_vcc_accum_v1.npz and it resolves at
#     /kaggle/input/datasets/<username>/<dataset-slug>/v5g0_vcc_accum_v1.npz
#     (same convention as the VCC probe run and the Replogle notebook); output
#     defaults to /kaggle/working when on Kaggle.
ROOT = "/sessions/peaceful-cool-bohr/mnt/Project 13"
_ON_KAGGLE = os.path.isdir("/kaggle/input")
_SANDBOX_RESDIR = os.path.join(ROOT, "results", "v5 G0 - VCC")
_KAGGLE_ACCUM_DEFAULT = ("/kaggle/input/datasets/ericdu847/v5g0-vcc-00-probe/"
                         "v5g0_vcc_accum_v1.npz")
RESDIR = os.environ.get("V5G0_OUT", "/kaggle/working" if _ON_KAGGLE else _SANDBOX_RESDIR)
os.makedirs(RESDIR, exist_ok=True)
ACCUM = os.environ.get(
    "V5G0_VCC_ACCUM",
    _KAGGLE_ACCUM_DEFAULT if _ON_KAGGLE else os.path.join(_SANDBOX_RESDIR, "v5g0_vcc_accum_v1.npz"))

# thresholds re-purposed from the original locked rule, now applied to the
# direct (undecomposed) reproducibility rather than to a specific-vs-global split
THRESH = [0.3, 0.6, 0.8]


def bid(gi, h, s, n_half, n_slot):
    return (gi * n_half + h) * n_slot + s


def corr(a, b):
    a = a - a.mean()
    b = b - b.mean()
    d = np.sqrt((a * a).sum() * (b * b).sum())
    return float((a * b).sum() / d) if d > 0 else np.nan


def pseudobulk_full(A, gi, half, gm, n_half, n_slot):
    """Sum ALL slots (0..n_slot-1) -> that group's full available half-depth,
    i.e. 'the depth the benchmark provides' for this specific perturbation."""
    s = A[bid(gi, half, 0, n_half, n_slot):bid(gi, half, n_slot - 1, n_half, n_slot) + 1].sum(axis=0)
    s = s[gm]
    tot = s.sum()
    return (np.log1p(s / tot * 1e6) if tot > 0 else None), int(tot > 0)


def pseudobulk_to_slot(A, gi, half, slot, gm, n_half, n_slot):
    s = A[bid(gi, half, 0, n_half, n_slot):bid(gi, half, slot, n_half, n_slot) + 1].sum(axis=0)
    s = s[gm]
    tot = s.sum()
    return np.log1p(s / tot * 1e6) if tot > 0 else None


def run():
    z = np.load(ACCUM, allow_pickle=True)
    accum, groups, avail = z["accum"], list(z["groups"]), z["avail"]
    run_kind, run_seed = list(z["run_kind"]), list(z["run_seed"])
    depths, detect, n_cells = list(z["depths"]), z["detect"], int(z["n_cells"])
    n_group, n_half, n_slot = int(z["n_group"]), int(z["n_half"]), int(z["n_slot"])
    ctrl = groups.index("non-targeting")
    gm = (detect / n_cells) >= 0.01
    n_deep_slot = len(depths) - 1   # slot index of the deepest fixed grid depth (600)

    real_runs = [i for i, k in enumerate(run_kind) if k == "real"]
    null_runs = [i for i, k in enumerate(run_kind) if k == "shuffled"]

    rows = []
    for ri in real_runs + null_runs:
        A = accum[ri]
        kind = run_kind[ri]
        seed = int(run_seed[ri])
        cA_full, _ = pseudobulk_full(A, ctrl, 0, gm, n_half, n_slot)
        cB_full, _ = pseudobulk_full(A, ctrl, 1, gm, n_half, n_slot)
        c600A = pseudobulk_to_slot(A, ctrl, 0, n_deep_slot, gm, n_half, n_slot)
        c600B = pseudobulk_to_slot(A, ctrl, 1, n_deep_slot, gm, n_half, n_slot)

        for gi in range(n_group):
            if gi == ctrl:
                continue
            n_own = int(min(avail[ri][gi, 0], avail[ri][gi, 1]))
            pA_full, ok = pseudobulk_full(A, gi, 0, gm, n_half, n_slot)
            pB_full, _ = pseudobulk_full(A, gi, 1, gm, n_half, n_slot)
            r_own = corr(pA_full - cA_full, pB_full - cB_full) if ok else np.nan

            r_600 = np.nan
            if avail[ri][gi, 0] >= 600 and avail[ri][gi, 1] >= 600:
                pA600 = pseudobulk_to_slot(A, gi, 0, n_deep_slot, gm, n_half, n_slot)
                pB600 = pseudobulk_to_slot(A, gi, 1, n_deep_slot, gm, n_half, n_slot)
                r_600 = corr(pA600 - c600A, pB600 - c600B)

            rows.append(dict(
                dataset="VCC_adata_Training", run_kind=kind, seed=seed,
                target=groups[gi], n_own_depth=n_own,
                r_direct_own_depth=r_own, r_direct_n600=r_600))

    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(RESDIR, "v5g0_vcc_direct_specific_v1.csv"), index=False)

    real = df[df.run_kind == "real"]
    null = df[df.run_kind == "shuffled"]
    per_target = real.groupby("target").r_direct_own_depth.mean()
    per_target_n = real.groupby("target").n_own_depth.mean()

    frac_above = {str(t): float((per_target >= t).mean()) for t in THRESH}
    n600 = real.dropna(subset=["r_direct_n600"])
    per_target_600 = n600.groupby("target").r_direct_n600.mean()
    frac_above_600 = {str(t): float((per_target_600 >= t).mean()) for t in THRESH}

    summary = {
        "task": "v5 G0 VCC — headline: fraction of perturbations with reproducible "
                "signal at the depth the benchmark provides",
        "ruling": "Brian 2026-07-19: gamma~0 on VCC -> decomposition degenerate, "
                  "s_p ~ d_p. Direct corr(d_p^A,d_p^B) replaces global/specific split.",
        "n_perturbations_total": int(len(per_target)),
        "own_depth_median_n": float(per_target_n.median()),
        "own_depth_cv": float(per_target_n.std() / per_target_n.mean()),
        "fraction_of_perturbations_above_threshold__own_achievable_depth": frac_above,
        "fraction_of_perturbations_above_threshold__matched_n600_only": frac_above_600,
        "n_perturbations_reaching_n600": int(len(per_target_600)),
        "shuffled_null_mean_r_direct": float(null.r_direct_own_depth.mean()),
        "distribution_percentiles_own_depth": {
            str(p): float(np.percentile(per_target, p)) for p in (5, 25, 50, 75, 95)
        },
        "reading": (
            "This is the number the field actually needs: at whatever depth VCC's "
            "150 perturbations naturally reach, what fraction reproduce well enough "
            "to be predictable in principle? Perturbations below ~0.3 cannot be "
            "modeled from this data at this depth regardless of model quality -- "
            "that's a property of the benchmark, not of any submitted model."),
    }
    with open(os.path.join(RESDIR, "v5g0_vcc_direct_specific_summary_v1.json"), "w") as f:
        json.dump(summary, f, indent=2)

    print(json.dumps(summary, indent=2))


if __name__ == "__main__":
    run()